# 03. Core Spatial Pipeline (Canonical)

This is the main branch to run for SICI + permutation + LOOCV + supplement exports.


## Why this branch

This branch was kept as the primary one because it is self-contained, has defensive checks, and includes table/figure exports.


In [7]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "GSE238264_RAW").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


Working directory: /Users/prateek/Downloads/Spatial HCC


In [8]:
# ============================
# NEXT BLOCK (paste + run)
# Fix torus NaN, patient-level permutation tests,
# baseline-vs-SICI + LOOCV ranking,
# and export supplement-ready tables + figures.
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, mannwhitneyu
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut



from pathlib import Path
import scanpy as sc

# Run-mode toggle: set HCC_RUN_MODE=smoke for quick checks.
RUN_MODE = os.environ.get("HCC_RUN_MODE", "full").strip().lower()
N_PERM_TORUS = 2000 if RUN_MODE == "full" else 100
N_PERM_LABEL = 20000 if RUN_MODE == "full" else 500

# Standalone load so this notebook can run from a fresh kernel.
if "st_adata" not in globals():
    h5_path = Path("st_adata_scored.h5ad")
    if not h5_path.exists():
        raise FileNotFoundError(
            "st_adata_scored.h5ad not found. Run 02_visium_spot_scoring.ipynb first "
            "or place the file in this working directory."
        )
    st_adata = sc.read_h5ad(h5_path.as_posix())

required_obs_cols = [
    "sample",
    "spot_exhaustion",
    "spot_cytotoxic",
    "spot_tam",
    "spot_caf",
    "spot_malignant",
    "spot_bcell",
    "spot_endo",
]
missing_obs_cols = [c for c in required_obs_cols if c not in st_adata.obs.columns]
if missing_obs_cols:
    raise KeyError(f"st_adata is missing required columns: {missing_obs_cols}")

# Bootstrap required objects if not already defined
if "st_by_sample" not in globals():
    if "st_adata" not in globals():
        raise NameError("Need `st_adata` in memory to build `st_by_sample`.")
    if "sample" not in st_adata.obs.columns:
        raise KeyError("`st_adata.obs['sample']` is required to build `st_by_sample`.")

    st_by_sample = {
        str(s): st_adata[st_adata.obs["sample"].astype(str) == str(s)].copy()
        for s in sorted(st_adata.obs["sample"].astype(str).unique())
    }

if "sample_groups" not in globals():
    sample_groups = {}

    # 1) Preferred source: per-spot response labels
    if "st_adata" in globals() and "response" in st_adata.obs.columns and "sample" in st_adata.obs.columns:
        resp_map = {
            "R": "R", "NR": "NR",
            "Responder": "R", "NonResponder": "NR",
            "responder": "R", "nonresponder": "NR",
            "non-responder": "NR", "non_responder": "NR",
        }
        tmp = (
            st_adata.obs[["sample", "response"]]
            .dropna(subset=["sample", "response"])
            .copy()
        )
        tmp["sample"] = tmp["sample"].astype(str)
        tmp["response"] = tmp["response"].astype(str)
        first_resp = tmp.groupby("sample", as_index=True)["response"].first()
        for s, r in first_resp.items():
            sample_groups[s] = resp_map.get(r, resp_map.get(r.strip(), r))

    # 2) Fallback: group_map variable from previous cells
    elif "group_map" in globals() and isinstance(group_map, dict):
        for s, g in group_map.items():
            g_str = str(g).strip().upper()
            if g_str == "NR":
                sample_groups[str(s)] = "NR"
            elif g_str == "R":
                sample_groups[str(s)] = "R"
            else:
                sample_groups[str(s)] = str(g)

    # 3) Last-resort heuristic from sample suffix
    for s in st_by_sample.keys():
        if s not in sample_groups:
            up = s.upper()
            if up.endswith("NR"):
                sample_groups[s] = "NR"
            elif up.endswith("R"):
                sample_groups[s] = "R"
            else:
                sample_groups[s] = None


# ----------------------------
# 0) REQUIRED INPUTS (you should already have these in your notebook)
# ----------------------------
# Expect:
#   st_by_sample: dict[str, AnnData]  # each AnnData is one slide/sample (e.g., "HCC1R")
#   sample_groups: dict[str, str]     # {"HCC1R":"R", ..., "HCC5NR":"NR"}
#
# Each AnnData should have:
#   - spatial coords in ad.obsm["spatial"] (Nx2)
#   - module fields in ad.obs, e.g.:
#       "spot_exhaustion", "spot_cytotoxic", "spot_tam", "spot_caf", "spot_malignant", "spot_bcell"
#
# Optional baselines (if you already computed them):
#   baseline_df with columns: ["sample", "mal_b_boundary_frac", ...]
#   sici_df with columns: ["sample", "SICI"]  (or compute from couplings below)


# ----------------------------
# 1) Grid indexing helper for torus shifts
#    (robust across visium-style coords / after concat)
# ----------------------------
def _grid_index_from_spatial(ad):
    """
    Returns integer (row, col) grid indices for each spot, for torus-shift.
    Preference order:
      1) ad.obs['array_row'], ad.obs['array_col'] if present
      2) derive from ad.obsm['spatial'] by scaling + rounding
    """
    if "array_row" in ad.obs.columns and "array_col" in ad.obs.columns:
        r = ad.obs["array_row"].to_numpy().astype(int)
        c = ad.obs["array_col"].to_numpy().astype(int)
        return r, c

    if "spatial" not in ad.obsm:
        raise KeyError("Need ad.obsm['spatial'] or ad.obs['array_row','array_col'].")

    xy = np.asarray(ad.obsm["spatial"], dtype=float)
    # Normalize scale so nearest-neighbor distances map ~1 grid step
    # Use median NN distance approx via pairwise on a sample (fast-ish)
    # Fallback: scale by median absolute diff in x
    x = xy[:, 0]
    y = xy[:, 1]

    # Robust step sizes (avoid full NN compute)
    dx = np.median(np.abs(np.diff(np.sort(x)))) if len(x) > 5 else 1.0
    dy = np.median(np.abs(np.diff(np.sort(y)))) if len(y) > 5 else 1.0
    dx = dx if np.isfinite(dx) and dx > 0 else 1.0
    dy = dy if np.isfinite(dy) and dy > 0 else 1.0

    xg = (x - np.nanmin(x)) / dx
    yg = (y - np.nanmin(y)) / dy

    c = np.rint(xg).astype(int)
    r = np.rint(yg).astype(int)
    return r, c


# ----------------------------
# 2) Torus-shift null correlation (safe)
#    You said you pasted this; keeping here so block is self-contained.
# ----------------------------
def torus_shift_null_corr_safe(ad, x_key, y_key, n_perm=2000, seed=1,
                               min_valid_frac=0.85, max_tries_factor=10):
    import squidpy as sq

    # Build once, then reuse spatial graph on repeated calls.
    if "spatial_connectivities" not in ad.obsp:
        sq.gr.spatial_neighbors(ad, coord_type="grid")
    W = ad.obsp["spatial_connectivities"]
    Wsum = np.maximum(W.sum(1).A1, 1e-9)

    x = ad.obs[x_key].to_numpy()
    y = ad.obs[y_key].to_numpy()

    x_lag = (W @ x) / Wsum
    obs = spearmanr(x_lag, y, nan_policy="omit")[0]

    r, c = _grid_index_from_spatial(ad)
    H = int(r.max() + 1)
    Wd = int(c.max() + 1)
    grid = np.full((H, Wd), np.nan, dtype=float)
    grid[r, c] = x

    rng = np.random.default_rng(seed)
    null = []
    tries = 0
    target = int(n_perm)
    max_tries = int(n_perm * max_tries_factor)

    base_valid = np.isfinite(y).sum()

    while len(null) < target and tries < max_tries:
        tries += 1
        dr = rng.integers(0, H)
        dc = rng.integers(0, Wd)
        shifted = np.roll(np.roll(grid, dr, axis=0), dc, axis=1)
        x_shift = shifted[r, c]

        valid = np.isfinite(x_shift) & np.isfinite(y)
        if valid.sum() < min_valid_frac * base_valid:
            continue

        xlag_p = (W @ x_shift) / Wsum
        rho = spearmanr(xlag_p[valid], y[valid], nan_policy="omit")[0]
        if np.isfinite(rho):
            null.append(rho)

    null = np.array(null, dtype=float)
    p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (len(null) + 1)
    return float(obs), float(p), int(len(null)), int(tries)


# ----------------------------
# 3) Compute couplings per sample (edge-gradient correlation)
# ----------------------------
def compute_edge_gradient_coupling(ad, key_a, key_b, graph="grid"):
    """
    Coupling C_{a,b} = corr(Δa_ij, Δb_ij) over edges (i,j) in spatial graph.
    Uses squidpy spatial graph; treats edges as undirected (i<j).
    """
    import squidpy as sq
    from scipy.sparse import triu

    if graph == "grid":
        if "spatial_connectivities" not in ad.obsp:
            sq.gr.spatial_neighbors(ad, coord_type="grid")
        A = ad.obsp["spatial_connectivities"]
    else:
        raise ValueError("graph must be 'grid' (keep consistent with torus).")

    # Upper triangle to avoid double-counting
    U = triu(A, k=1).tocoo()
    i = U.row
    j = U.col

    a = ad.obs[key_a].to_numpy()
    b = ad.obs[key_b].to_numpy()
    da = a[i] - a[j]
    db = b[i] - b[j]

    valid = np.isfinite(da) & np.isfinite(db)
    if valid.sum() < 10:
        return np.nan

    rho = spearmanr(da[valid], db[valid], nan_policy="omit")[0]
    return float(rho)


# ----------------------------
# 4) Run torus-shift + couplings
# ----------------------------
# Reviewer-required full matrix for Q2/W3 (17 pairs, including MAL-CAF).
pairwise_coupling_specs = [
    ("EXH-CYTO", "spot_exhaustion", "spot_cytotoxic", "C_exh_cyto"),
    ("EXH-TAM",  "spot_exhaustion", "spot_tam",       "C_exh_tam"),
    ("EXH-CAF",  "spot_exhaustion", "spot_caf",       "C_exh_caf"),
    ("EXH-B",    "spot_exhaustion", "spot_bcell",     "C_exh_b"),
    ("EXH-MAL",  "spot_exhaustion", "spot_malignant", "C_exh_malig"),
    ("CYTO-TAM", "spot_cytotoxic",  "spot_tam",       "C_cyto_tam"),
    ("CYTO-CAF", "spot_cytotoxic",  "spot_caf",       "C_cyto_caf"),
    ("CYTO-B",   "spot_bcell",      "spot_cytotoxic", "C_b_cyto"),
    ("CYTO-MAL", "spot_cytotoxic",  "spot_malignant", "C_cyto_malig"),
    ("B-TAM",    "spot_bcell",      "spot_tam",       "C_b_tam"),
    ("B-CAF",    "spot_bcell",      "spot_caf",       "C_b_caf"),
    ("B-MAL",    "spot_bcell",      "spot_malignant", "C_b_malig"),
    ("TAM-CAF",  "spot_tam",        "spot_caf",       "C_tam_caf"),
    ("TAM-MAL",  "spot_tam",        "spot_malignant", "C_tam_malig"),
    ("CAF-ENDO", "spot_caf",        "spot_endo",      "C_caf_endo"),
    ("MAL-ENDO", "spot_malignant",  "spot_endo",      "C_malig_endo"),
    ("MAL-CAF",  "spot_malignant",  "spot_caf",       "C_malig_caf"),
]
pairwise_coupling_metrics = [metric for _, _, _, metric in pairwise_coupling_specs]
pairwise_metric_to_pair = {metric: pair for pair, _, _, metric in pairwise_coupling_specs}

# torus test targets (choose what you want to claim as "spatial-null checked")
# Here I include EXH ~ lag(TAM) and EXH ~ lag(CYTO) (you can add more)
torus_tests = [
    ("spot_tam",        "spot_exhaustion", "torus_TAMlag_vs_EXH"),
    ("spot_cytotoxic",  "spot_exhaustion", "torus_CYTOlag_vs_EXH"),
]

rows = []
for sample, ad in st_by_sample.items():
    row = {"sample": sample, "group": sample_groups.get(sample, None)}

    # Couplings
    for _, a, b, name in pairwise_coupling_specs:
        row[name] = compute_edge_gradient_coupling(ad, a, b, graph="grid")

    # Torus-shift nulls (safe)
    for x_key, y_key, name in torus_tests:
        rho_obs, p_torus, n_eff, tries = torus_shift_null_corr_safe(
            ad, x_key=x_key, y_key=y_key,
            n_perm=N_PERM_TORUS, seed=1, min_valid_frac=0.85
        )
        row[f"{name}_rho_obs"] = rho_obs
        row[f"{name}_p_torus"] = p_torus
        row[f"{name}_n_eff_perm"] = n_eff
        row[f"{name}_tries"] = tries

    rows.append(row)

metrics_df = pd.DataFrame(rows).sort_values("sample").reset_index(drop=True)

# Keep only clean binary response labels for patient-level tests.
metrics_df["group"] = (
    metrics_df["group"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"RESPONDER": "R", "NONRESPONDER": "NR", "NON-RESPONDER": "NR", "NON_RESPONDER": "NR"})
)
metrics_df = metrics_df[metrics_df["group"].isin(["R", "NR"])].copy()
if metrics_df.empty or set(metrics_df["group"].unique()) != {"R", "NR"}:
    raise ValueError("Need both R and NR samples after group-label normalization.")

display(metrics_df)


# ----------------------------
# 5) Define / recompute SICI (a priori fixed weights)
#    Use your fixed coefficients (edit if your manuscript uses different).
# ----------------------------
alpha, beta, gamma = 1.0, 1.0, 1.0  # keep fixed; document as a priori
metrics_df["SICI"] = (
    alpha * metrics_df["C_b_cyto"]
    + beta  * metrics_df["C_malig_caf"]
    + gamma * metrics_df["C_exh_cyto"]
)

display(metrics_df[["sample","group","SICI","C_b_cyto","C_malig_caf","C_exh_cyto"]])


# ----------------------------
# 6) Patient-level label permutation tests (R vs NR) on aggregated metrics
#    Stats: difference in means (R - NR)
# ----------------------------
def label_permutation_test(df, value_col, group_col="group", n_perm=20000, seed=1):
    """Permutation test of mean difference (R - NR) at patient level."""
    rng = np.random.default_rng(seed)
    x = df[[value_col, group_col]].dropna()
    vals = x[value_col].to_numpy()
    grp  = x[group_col].to_numpy()

    if set(np.unique(grp)) != {"R","NR"}:
        raise ValueError(f"{value_col}: need both groups 'R' and 'NR' present.")

    obs = vals[grp=="R"].mean() - vals[grp=="NR"].mean()

    null = []
    for _ in range(n_perm):
        grp_perm = rng.permutation(grp)
        stat = vals[grp_perm=="R"].mean() - vals[grp_perm=="NR"].mean()
        null.append(stat)
    null = np.asarray(null, dtype=float)

    p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (len(null) + 1)
    return float(obs), float(p), null

def add_bh_fdr(df, p_col="p_perm", q_col="q_fdr_bh", reject_col="reject_fdr_0p05", alpha=0.05):
    """Benjamini-Hochberg FDR correction for a p-value column."""
    out = df.copy()
    p = out[p_col].to_numpy(dtype=float)
    valid = np.isfinite(p)

    q = np.full(len(out), np.nan, dtype=float)
    reject = np.full(len(out), False, dtype=bool)

    if valid.any():
        pv = p[valid]
        m = len(pv)
        order = np.argsort(pv)
        ranked = pv[order]
        q_ranked = ranked * m / np.arange(1, m + 1)
        q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
        q_ranked = np.clip(q_ranked, 0.0, 1.0)

        qv = np.empty(m, dtype=float)
        qv[order] = q_ranked
        q[valid] = qv
        reject[valid] = qv <= alpha

    out[q_col] = q
    out[reject_col] = reject
    return out

def effect_sizes_by_group(df, value_col, group_col="group"):
    """Compute Cohen's d and Cliff's delta (via Mann-Whitney U) for R vs NR."""
    x = df[[value_col, group_col]].dropna()
    xr = x.loc[x[group_col] == "R", value_col].to_numpy(dtype=float)
    xnr = x.loc[x[group_col] == "NR", value_col].to_numpy(dtype=float)

    n_r, n_nr = len(xr), len(xnr)
    if n_r == 0 or n_nr == 0:
        return {
            "cohens_d": np.nan,
            "cliffs_delta": np.nan,
            "mannwhitney_u": np.nan,
            "p_mannwhitney": np.nan,
            "C_all": np.nan,
        }

    s_r = np.std(xr, ddof=1) if n_r > 1 else np.nan
    s_nr = np.std(xnr, ddof=1) if n_nr > 1 else np.nan
    denom = np.sqrt(((n_r - 1) * (s_r ** 2) + (n_nr - 1) * (s_nr ** 2)) / max(n_r + n_nr - 2, 1))
    cohens_d = (np.mean(xr) - np.mean(xnr)) / denom if np.isfinite(denom) and denom > 0 else np.nan

    try:
        mwu = mannwhitneyu(xr, xnr, alternative="two-sided", method="auto")
        u_stat = float(mwu.statistic)
        p_mwu = float(mwu.pvalue)
        cliffs_delta = (2.0 * u_stat) / (n_r * n_nr) - 1.0
    except Exception:
        u_stat = np.nan
        p_mwu = np.nan
        cliffs_delta = np.nan

    return {
        "cohens_d": float(cohens_d) if np.isfinite(cohens_d) else np.nan,
        "cliffs_delta": float(cliffs_delta) if np.isfinite(cliffs_delta) else np.nan,
        "mannwhitney_u": u_stat,
        "p_mannwhitney": p_mwu,
        "C_all": float(np.nanmean(x[value_col].to_numpy(dtype=float))),
    }

# Reviewer-required: 20,000 label permutations per metric.
N_PERM_LABEL_DETAILED = 20000

perm_cols = ["SICI"] + pairwise_coupling_metrics
perm_cols = list(dict.fromkeys(perm_cols))
perm_rows = []
perm_nulls = {}  # store for optional plots
for col in perm_cols:
    obs, p, null = label_permutation_test(metrics_df, col, n_perm=N_PERM_LABEL_DETAILED, seed=1)
    eff = effect_sizes_by_group(metrics_df, col)
    perm_rows.append(
        {
            "metric": col,
            "diff_mean_R_minus_NR": obs,
            "p_perm": p,
            **eff,
        }
    )
    perm_nulls[col] = null

perm_df = pd.DataFrame(perm_rows)
perm_df = add_bh_fdr(perm_df, p_col="p_perm", q_col="q_fdr_bh", reject_col="reject_fdr_0p05", alpha=0.05)
perm_df = perm_df.sort_values("p_perm").reset_index(drop=True)
display(perm_df)

# Dedicated reviewer table: full detailed pairwise coupling matrix.
# BH family here is all 17 pairwise tests, including MAL-CAF.
full_detailed_perm_df = perm_df[perm_df["metric"].isin(pairwise_coupling_metrics)].copy()
full_detailed_perm_df = add_bh_fdr(
    full_detailed_perm_df,
    p_col="p_perm",
    q_col="q_fdr_bh",
    reject_col="reject_fdr_0p05",
    alpha=0.05,
)
full_detailed_perm_df["pair"] = full_detailed_perm_df["metric"].map(pairwise_metric_to_pair)
full_detailed_perm_df = full_detailed_perm_df[
    [
        "pair",
        "metric",
        "C_all",
        "diff_mean_R_minus_NR",
        "cohens_d",
        "cliffs_delta",
        "mannwhitney_u",
        "p_mannwhitney",
        "p_perm",
        "q_fdr_bh",
        "reject_fdr_0p05",
    ]
].sort_values(["p_perm", "pair"]).reset_index(drop=True)
display(full_detailed_perm_df)


# ----------------------------
# 7) Baseline-vs-SICI comparison + LOOCV ranking
# ----------------------------
# If you already computed baseline_df earlier, it should contain "sample" and baseline columns.
# If not, create a placeholder and skip.
def _compute_simple_baselines(ad):
    """Compute baseline features used for baseline-vs-SICI comparisons."""
    import squidpy as sq

    sq.gr.spatial_neighbors(ad, coord_type="grid")
    W = ad.obsp["spatial_connectivities"]

    cyto = ad.obs["spot_cytotoxic"].to_numpy()
    mal = ad.obs["spot_malignant"].to_numpy()
    b = ad.obs["spot_bcell"].to_numpy()

    mal_hi = mal > np.quantile(mal, 0.75)
    cyto_in_mal_hi = float(np.nanmean(cyto[mal_hi]))

    Wsum = np.maximum(W.sum(1).A1, 1e-9)
    cyto_lag = (W @ cyto) / Wsum
    nbr_cyto_around_mal_hi = float(np.nanmean(cyto_lag[mal_hi]))

    coo = W.tocoo()
    mask = coo.row < coo.col
    i = coo.row[mask]
    j = coo.col[mask]
    b_sign = np.sign(b)
    m_sign = np.sign(mal)
    mal_b_boundary_frac = float(np.mean((b_sign[i] * m_sign[j]) < 0)) if len(i) else np.nan

    return {
        "cyto_in_mal_hi": cyto_in_mal_hi,
        "nbr_cyto_around_mal_hi": nbr_cyto_around_mal_hi,
        "mal_b_boundary_frac": mal_b_boundary_frac,
    }

if "baseline_df" in globals() and isinstance(baseline_df, pd.DataFrame):
    base = baseline_df.copy()
else:
    baseline_rows = []
    for sample, ad in st_by_sample.items():
        row = {"sample": sample}
        row.update(_compute_simple_baselines(ad))
        baseline_rows.append(row)
    base = pd.DataFrame(baseline_rows).sort_values("sample").reset_index(drop=True)
    baseline_df = base.copy()

y = metrics_df["group"].map({"NR":0, "R":1}).to_numpy()

# Join baseline columns (if available)
eval_df = metrics_df[["sample","group","SICI"]].copy()
if base is not None:
    eval_df = eval_df.merge(base, on="sample", how="left")

display(eval_df)

def loocv_logreg_single_feature(df, feature, y_col="group"):
    """Single-feature LOOCV logistic regression with AUC/ACC/BalACC outputs."""
    d = df[[feature, y_col]].dropna().copy()
    X = d[[feature]].to_numpy().astype(float)
    y = d[y_col].map({"NR":0,"R":1}).to_numpy().astype(int)

    loo = LeaveOneOut()
    probs = np.zeros(len(d), dtype=float)
    preds = np.zeros(len(d), dtype=int)

    for tr, te in loo.split(X):
        model = LogisticRegression(solver="lbfgs", C=1.0, max_iter=1000)
        model.fit(X[tr], y[tr])
        probs[te] = model.predict_proba(X[te])[:,1]
        preds[te] = (probs[te] >= 0.5).astype(int)

    # AUC needs both classes present in d (true here)
    auc = roc_auc_score(y, probs) if len(np.unique(y)) == 2 else np.nan
    acc = accuracy_score(y, preds)
    bacc = balanced_accuracy_score(y, preds)
    return float(auc), float(acc), float(bacc), d.index.to_numpy(), probs, preds

feature_list = ["SICI"]
preferred_baselines = ["cyto_in_mal_hi", "nbr_cyto_around_mal_hi", "mal_b_boundary_frac"]
for col in preferred_baselines:
    if col in base.columns:
        feature_list.append(col)
for col in base.columns:
    if col != "sample" and col not in feature_list:
        feature_list.append(col)

rank_rows = []
loocv_store = {}
for feat in feature_list:
    auc, acc, bacc, idx, probs, preds = loocv_logreg_single_feature(eval_df, feat, y_col="group")
    rank_rows.append({"feature": feat, "LOOCV_AUC": auc, "LOOCV_ACC": acc, "LOOCV_BalACC": bacc})
    loocv_store[feat] = {"idx": idx, "probs": probs, "preds": preds}

rank_df = pd.DataFrame(rank_rows).sort_values(["LOOCV_AUC","LOOCV_BalACC"], ascending=False).reset_index(drop=True)
display(rank_df)


# ----------------------------
# 8) Export “Supplement-ready” tables
# ----------------------------
out_dir = "latest_figures"
import os
os.makedirs(out_dir, exist_ok=True)

metrics_df.to_csv(f"{out_dir}/Table_S1_metrics_couplings_torus.csv", index=False)
perm_df.to_csv(f"{out_dir}/Table_S2_label_permutation_tests.csv", index=False)
rank_df.to_csv(f"{out_dir}/Table_S3_LOOCV_baseline_vs_SICI.csv", index=False)
full_detailed_perm_df.to_csv(f"{out_dir}/Table_S_full_detailed_coupling.csv", index=False)

print("Wrote:")
print(f" - {out_dir}/Table_S1_metrics_couplings_torus.csv")
print(f" - {out_dir}/Table_S2_label_permutation_tests.csv")
print(f" - {out_dir}/Table_S3_LOOCV_baseline_vs_SICI.csv")
print(f" - {out_dir}/Table_S_full_detailed_coupling.csv")


# ----------------------------
# 9) Figures saving commands (minimal, journal-friendly)
# ----------------------------

# Fig S1: SICI by group (strip + mean)
plt.figure()
for g in ["NR","R"]:
    d = metrics_df.loc[metrics_df["group"]==g, "SICI"].dropna().to_numpy()
    x = np.full_like(d, 0 if g=="NR" else 1, dtype=float)
    plt.scatter(x, d)
plt.xticks([0,1], ["NR","R"])
plt.ylabel("SICI")
plt.title("SICI by response group")
plt.tight_layout()
plt.savefig(f"{out_dir}/Fig_S1_SICI_by_group.png", dpi=300)
plt.close()

# Fig S2: C_malig_caf by group (strip + mean)
plt.figure()
for g in ["NR","R"]:
    d = metrics_df.loc[metrics_df["group"]==g, "C_malig_caf"].dropna().to_numpy()
    x = np.full_like(d, 0 if g=="NR" else 1, dtype=float)
    plt.scatter(x, d)
plt.xticks([0,1], ["NR","R"])
plt.ylabel("C_malig_caf")
plt.title("C_malig_caf by response group")
plt.tight_layout()
plt.savefig(f"{out_dir}/Fig_S2_C_malig_caf_by_group.png", dpi=300)
plt.close()

# Fig S2b: Permutation null histogram for SICI (patient-label permutation)
if "SICI" in perm_nulls:
    plt.figure()
    plt.hist(perm_nulls["SICI"], bins=40)
    obs = perm_df.loc[perm_df["metric"]=="SICI","diff_mean_R_minus_NR"].values
    if len(obs):
        plt.axvline(obs[0])
    plt.xlabel("Null: mean(R)-mean(NR)")
    plt.ylabel("count")
    plt.title("Label permutation null for SICI")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/Fig_S2_permnull_SICI.png", dpi=300)
    plt.close()

# Fig S3: Scale-robustness plot for C_exh_cyto (if you still have your scale table as df_scale)
# Expected df_scale columns: ["sample","scale","C_exh_cyto"]
if "df_scale" in globals():
    plt.figure(figsize=(7,4))
    for s, sub in df_scale.groupby("sample"):
        plt.plot(sub["scale"], sub["C_exh_cyto"], marker="o", linewidth=1)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("C_exh_cyto")
    plt.title("C_exh_cyto across scales")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/Fig_S3_scale_robustness_C_exh_cyto.png", dpi=300)
    plt.close()

print("Saved figures (PNG) into:", out_dir)


,sample,group,C_exh_cyto,C_exh_tam,C_exh_caf,C_exh_b,C_exh_malig,C_cyto_tam,C_cyto_caf,C_b_cyto,...,C_malig_endo,C_malig_caf,torus_TAMlag_vs_EXH_rho_obs,torus_TAMlag_vs_EXH_p_torus,torus_TAMlag_vs_EXH_n_eff_perm,torus_TAMlag_vs_EXH_tries,torus_CYTOlag_vs_EXH_rho_obs,torus_CYTOlag_vs_EXH_p_torus,torus_CYTOlag_vs_EXH_n_eff_perm,torus_CYTOlag_vs_EXH_tries
0,HCC1R,R,0.019049,-0.002871,-0.014660,0.004304,-0.031428,0.003096,0.002404,0.017711,...,-0.042088,-0.033758,-0.124861,0.064332,1398,20000,-0.030197,0.318799,1398,20000
1,HCC2R,R,0.038262,-0.008495,0.003847,0.010747,0.002953,-0.017909,-0.001325,-0.032210,...,0.022006,-0.057423,-0.115302,0.335106,187,20000,-0.044110,0.500000,187,20000
2,HCC3R,R,0.022126,0.029213,-0.012453,0.017455,-0.032975,0.014950,-0.001883,0.007831,...,-0.017079,-0.040782,-0.145252,0.546667,74,20000,0.056219,0.400000,74,20000
3,HCC4R,R,0.061939,0.024218,0.005182,0.084225,-0.051629,0.067595,0.011912,0.095418,...,0.017078,-0.035658,0.059061,0.402439,81,20000,0.060354,0.390244,81,20000
4,HCC5NR,NR,-0.025713,0.035130,0.008003,0.026218,-0.053142,0.044151,0.083040,-0.010910,...,0.005172,-0.065038,0.122185,0.276596,46,20000,0.031427,0.191489,46,20000
5,HCC6NR,NR,0.032672,-0.010161,-0.001378,-0.012342,-0.001000,0.031591,0.005949,0.002226,...,-0.038438,-0.104665,0.016974,0.626866,66,20000,0.054720,0.313433,66,20000
6,HCC7NR,NR,0.010254,0.029231,0.024375,-0.038641,-0.016060,0.020775,0.078866,0.032413,...,0.001921,-0.070682,0.063514,0.280000,24,20000,0.076721,0.480000,24,20000


,sample,group,SICI,C_b_cyto,C_malig_caf,C_exh_cyto
0,HCC1R,R,0.003003,0.017711,-0.033758,0.019049
1,HCC2R,R,-0.051372,-0.032210,-0.057423,0.038262
2,HCC3R,R,-0.010825,0.007831,-0.040782,0.022126
3,HCC4R,R,0.121699,0.095418,-0.035658,0.061939
4,HCC5NR,NR,-0.101660,-0.010910,-0.065038,-0.025713
5,HCC6NR,NR,-0.069766,0.002226,-0.104665,0.032672
6,HCC7NR,NR,-0.028015,0.032413,-0.070682,0.010254


,metric,diff_mean_R_minus_NR,p_perm,cohens_d,cliffs_delta,mannwhitney_u,p_mannwhitney,C_all,q_fdr_bh,reject_fdr_0p05
0,C_malig_caf,0.038223,0.026949,2.401662,1.000000,12.0,0.057143,-0.058287,0.477876,False
1,C_cyto_caf,-0.053174,0.053097,-1.908547,-0.833333,1.0,0.114286,0.025566,0.477876,False
2,SICI,0.082107,0.138593,1.320627,0.833333,11.0,0.114286,-0.019562,0.655424,False
3,C_exh_caf,-0.014854,0.196440,-1.283529,-0.666667,2.0,0.228571,0.001845,0.655424,False
4,C_exh_cyto,0.029606,0.197890,1.231222,0.666667,10.0,0.228571,0.022656,0.655424,False
5,C_tam_caf,-0.024038,0.226489,-1.010988,-0.666667,2.0,0.228571,0.067103,0.655424,False
6,C_exh_b,0.037438,0.254887,1.058533,0.500000,9.0,0.400000,0.013138,0.655424,False
7,C_tam_malig,-0.045471,0.312034,-0.833261,-0.333333,4.0,0.628571,-0.004226,0.702077,False
8,C_cyto_tam,-0.015239,0.516274,-0.522726,-0.500000,3.0,0.400000,0.023464,0.877615,False
9,C_exh_tam,-0.007550,0.683616,-0.352759,-0.333333,4.0,0.628571,0.013752,0.877615,False


,pair,metric,C_all,diff_mean_R_minus_NR,cohens_d,cliffs_delta,mannwhitney_u,p_mannwhitney,p_perm,q_fdr_bh,reject_fdr_0p05
0,MAL-CAF,C_malig_caf,-0.058287,0.038223,2.401662,1.000000,12.0,0.057143,0.026949,0.451327,False
1,CYTO-CAF,C_cyto_caf,0.025566,-0.053174,-1.908547,-0.833333,1.0,0.114286,0.053097,0.451327,False
2,EXH-CAF,C_exh_caf,0.001845,-0.014854,-1.283529,-0.666667,2.0,0.228571,0.196440,0.722181,False
3,EXH-CYTO,C_exh_cyto,0.022656,0.029606,1.231222,0.666667,10.0,0.228571,0.197890,0.722181,False
4,TAM-CAF,C_tam_caf,0.067103,-0.024038,-1.010988,-0.666667,2.0,0.228571,0.226489,0.722181,False
5,EXH-B,C_exh_b,0.013138,0.037438,1.058533,0.500000,9.0,0.400000,0.254887,0.722181,False
6,TAM-MAL,C_tam_malig,-0.004226,-0.045471,-0.833261,-0.333333,4.0,0.628571,0.312034,0.757798,False
7,CYTO-TAM,C_cyto_tam,0.023464,-0.015239,-0.522726,-0.500000,3.0,0.400000,0.516274,0.880662,False
8,EXH-TAM,C_exh_tam,0.013752,-0.007550,-0.352759,-0.333333,4.0,0.628571,0.683616,0.880662,False
9,CAF-ENDO,C_caf_endo,0.039294,-0.012686,-0.261048,-0.333333,4.0,0.628571,0.708515,0.880662,False


,sample,group,SICI,cyto_in_mal_hi,nbr_cyto_around_mal_hi,mal_b_boundary_frac
0,HCC1R,R,0.003003,0.028961,0.025292,0.080474
1,HCC2R,R,-0.051372,0.026753,0.026994,0.227956
2,HCC3R,R,-0.010825,0.001542,0.007846,0.195931
3,HCC4R,R,0.121699,-0.015853,-0.009954,0.067111
4,HCC5NR,NR,-0.101660,-0.001244,0.002124,0.449083
5,HCC6NR,NR,-0.069766,-0.061175,-0.061292,0.840963
6,HCC7NR,NR,-0.028015,-0.036813,-0.036414,0.373465


,feature,LOOCV_AUC,LOOCV_ACC,LOOCV_BalACC
0,mal_b_boundary_frac,0.0,0.571429,0.500
1,SICI,0.0,0.428571,0.375
2,cyto_in_mal_hi,0.0,0.428571,0.375
3,nbr_cyto_around_mal_hi,0.0,0.428571,0.375


Wrote:
 - latest_figures/Table_S1_metrics_couplings_torus.csv
 - latest_figures/Table_S2_label_permutation_tests.csv
 - latest_figures/Table_S3_LOOCV_baseline_vs_SICI.csv
 - latest_figures/Table_S_full_detailed_coupling.csv
Saved figures (PNG) into: latest_figures


In [9]:
import numpy as np
import matplotlib.pyplot as plt

target_metrics = ["SICI", "C_malig_caf"]

RUN_MODE = os.environ.get("HCC_RUN_MODE", "full").strip().lower()
N_PERM_PLOT = 50000 if RUN_MODE == "full" else 1000

def _obs_diff_mean_r_minus_nr(df, value_col, group_col="group"):
    """Observed patient-level mean difference: responders minus non-responders."""
    x = df[[value_col, group_col]].dropna()
    vals = x[value_col].to_numpy()
    grp = x[group_col].to_numpy()
    return float(vals[grp=="R"].mean() - vals[grp=="NR"].mean())

def _perm_null_from_metrics(df, value_col, group_col="group", n_perm=50000, seed=1):
    """Build a permutation null for patient-level mean difference (R - NR)."""
    rng = np.random.default_rng(seed)
    x = df[[value_col, group_col]].dropna()
    vals = x[value_col].to_numpy()
    grp = x[group_col].to_numpy()
    null = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        gp = rng.permutation(grp)
        null[i] = vals[gp=="R"].mean() - vals[gp=="NR"].mean()
    obs = float(vals[grp=="R"].mean() - vals[grp=="NR"].mean())
    return obs, null

def plot_perm_null(perm_vals, obs_val, title, outname):
    """Plot and save permutation-null histogram with observed statistic marker."""
    plt.figure(figsize=(6,4))
    plt.hist(perm_vals, bins=30)
    plt.axvline(obs_val, linestyle="--")
    plt.title(title)
    plt.xlabel("Permuted diff (mean_R - mean_NR)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(outname, dpi=300)
    plt.close()

perm_arrays = {}
obs_vals = {}

if "perm_nulls" in globals() and isinstance(perm_nulls, dict):
    for m in target_metrics:
        if m in perm_nulls:
            perm_arrays[m] = np.asarray(perm_nulls[m], dtype=float)

if "perm2_df" in globals():
    for m in target_metrics:
        v = perm2_df.loc[perm2_df["metric"] == m, "diff_mean_R_minus_NR"].to_numpy()
        if v.size:
            obs_vals[m] = float(v[0])

if "perm_df" in globals():
    for m in target_metrics:
        v = perm_df.loc[perm_df["metric"] == m, "diff_mean_R_minus_NR"].to_numpy()
        if v.size and m not in obs_vals:
            obs_vals[m] = float(v[0])

for m in target_metrics:
    if m not in perm_arrays:
        if "metrics_df" not in globals():
            raise RuntimeError("metrics_df not found. Run the metric/permutation cells first.")
        obs_calc, null_calc = _perm_null_from_metrics(metrics_df, m, n_perm=N_PERM_PLOT, seed=1)
        perm_arrays[m] = null_calc
        if m not in obs_vals:
            obs_vals[m] = obs_calc
    elif m not in obs_vals:
        if "metrics_df" not in globals():
            raise RuntimeError("Observed stat missing and metrics_df not found.")
        obs_vals[m] = _obs_diff_mean_r_minus_nr(metrics_df, m)

plot_perm_null(
    perm_arrays["SICI"],
    obs_vals["SICI"],
    "Patient-label permutation null: SICI",
    "fig_permnull_SICI.png",
)

plot_perm_null(
    perm_arrays["C_malig_caf"],
    obs_vals["C_malig_caf"],
    "Patient-label permutation null: C_malig_caf",
    "fig_permnull_C_malig_caf.png",
)

print("Saved: fig_permnull_SICI.png, fig_permnull_C_malig_caf.png")

Saved: fig_permnull_SICI.png, fig_permnull_C_malig_caf.png


In [10]:
# ----------------------------
# Q5/W9) Malignant-restricted recomputation
# ----------------------------
# 1) Restrict each section to spots with malignant score > 0
# 2) Rebuild spatial graph on masked spots
# 3) Recompute 4 core couplings + SICI
# 4) Check direction preservation vs full tissue

import numpy as np
import pandas as pd
import squidpy as sq

if "st_by_sample" not in globals() or "metrics_df" not in globals():
    raise RuntimeError("Run the main core pipeline cell first to create st_by_sample and metrics_df.")

core_specs = [
    ("C_exh_cyto", "spot_exhaustion", "spot_cytotoxic"),
    ("C_exh_tam", "spot_exhaustion", "spot_tam"),
    ("C_malig_caf", "spot_malignant", "spot_caf"),
    ("C_b_cyto", "spot_bcell", "spot_cytotoxic"),
]

alpha_use = globals().get("alpha", 1.0)
beta_use = globals().get("beta", 1.0)
gamma_use = globals().get("gamma", 1.0)

rows = []
for sample, ad in st_by_sample.items():
    mal = ad.obs["spot_malignant"].to_numpy(dtype=float)
    mask = np.isfinite(mal) & (mal > 0)
    ad_mask = ad[mask].copy()

    row = {
        "sample": sample,
        "group": sample_groups.get(sample, None),
        "n_spots_full": int(ad.n_obs),
        "n_spots_malignant_gt0": int(mask.sum()),
    }

    if ad_mask.n_obs < 10:
        for metric, _, _ in core_specs:
            row[metric] = np.nan
    else:
        sq.gr.spatial_neighbors(ad_mask, coord_type="grid")
        for metric, key_a, key_b in core_specs:
            row[metric] = compute_edge_gradient_coupling(ad_mask, key_a, key_b, graph="grid")

    if np.isfinite(row["C_b_cyto"]) and np.isfinite(row["C_malig_caf"]) and np.isfinite(row["C_exh_cyto"]):
        row["SICI"] = (
            alpha_use * row["C_b_cyto"]
            + beta_use * row["C_malig_caf"]
            + gamma_use * row["C_exh_cyto"]
        )
    else:
        row["SICI"] = np.nan

    rows.append(row)

malignant_restricted_df = pd.DataFrame(rows).sort_values("sample").reset_index(drop=True)

full_cols = ["sample", "group", "C_exh_cyto", "C_exh_tam", "C_malig_caf", "C_b_cyto", "SICI"]
full_core_df = metrics_df[full_cols].copy()

malignant_restricted_df = malignant_restricted_df.merge(
    full_core_df,
    on=["sample", "group"],
    how="left",
    suffixes=("_malignant_restricted", "_full"),
)

for metric in ["C_exh_cyto", "C_exh_tam", "C_malig_caf", "C_b_cyto", "SICI"]:
    rcol = f"{metric}_malignant_restricted"
    fcol = f"{metric}_full"
    dcol = f"delta_{metric}_restricted_minus_full"
    pcol = f"direction_preserved_{metric}"

    malignant_restricted_df[dcol] = malignant_restricted_df[rcol] - malignant_restricted_df[fcol]

    valid = np.isfinite(malignant_restricted_df[rcol]) & np.isfinite(malignant_restricted_df[fcol])
    dir_series = pd.Series([pd.NA] * len(malignant_restricted_df), dtype="boolean")
    dir_series.loc[valid] = (
        np.sign(malignant_restricted_df.loc[valid, rcol]) == np.sign(malignant_restricted_df.loc[valid, fcol])
    )
    malignant_restricted_df[pcol] = dir_series

display(malignant_restricted_df)

preservation_summary = []
for metric in ["C_exh_cyto", "C_exh_tam", "C_malig_caf", "C_b_cyto", "SICI"]:
    pcol = f"direction_preserved_{metric}"
    vals = malignant_restricted_df[pcol].map({True: 1.0, False: 0.0})
    preservation_summary.append(
        {
            "metric": metric,
            "fraction_direction_preserved": float(np.nanmean(vals.to_numpy(dtype=float))) if vals.notna().any() else np.nan,
        }
    )
preservation_summary_df = pd.DataFrame(preservation_summary)
display(preservation_summary_df)

out_dir_local = globals().get("out_dir", "latest_figures")
import os
os.makedirs(out_dir_local, exist_ok=True)
malignant_restricted_df.to_csv(f"{out_dir_local}/Table_S_malignant_restricted.csv", index=False)
print(f"Wrote: {out_dir_local}/Table_S_malignant_restricted.csv")


INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           


,sample,group,n_spots_full,n_spots_malignant_gt0,C_exh_cyto_malignant_restricted,C_exh_tam_malignant_restricted,C_malig_caf_malignant_restricted,C_b_cyto_malignant_restricted,SICI_malignant_restricted,C_exh_cyto_full,...,delta_C_exh_cyto_restricted_minus_full,direction_preserved_C_exh_cyto,delta_C_exh_tam_restricted_minus_full,direction_preserved_C_exh_tam,delta_C_malig_caf_restricted_minus_full,direction_preserved_C_malig_caf,delta_C_b_cyto_restricted_minus_full,direction_preserved_C_b_cyto,delta_SICI_restricted_minus_full,direction_preserved_SICI
0,HCC1R,R,3006,2727,0.012700,0.003039,-0.026055,0.024816,0.011460,0.019049,...,-0.006349,True,0.005910,False,0.007703,True,0.007104,True,0.008457,True
1,HCC2R,R,2766,2651,0.039070,-0.001708,-0.075353,-0.029301,-0.065584,0.038262,...,0.000808,True,0.006788,True,-0.017929,True,0.002909,True,-0.014212,True
2,HCC3R,R,2170,2170,0.022126,0.029213,-0.040782,0.007831,-0.010825,0.022126,...,0.000000,True,0.000000,True,0.000000,True,0.000000,True,0.000000,True
3,HCC4R,R,3002,3002,0.061939,0.024218,-0.035658,0.095418,0.121699,0.061939,...,0.000000,True,0.000000,True,0.000000,True,0.000000,True,0.000000,True
4,HCC5NR,NR,1320,1307,-0.027586,0.032588,-0.055265,-0.010511,-0.093362,-0.025713,...,-0.001874,True,-0.002542,True,0.009773,True,0.000399,True,0.008298,True
5,HCC6NR,NR,2575,2574,0.032482,-0.009858,-0.105200,0.003840,-0.068878,0.032672,...,-0.000191,True,0.000303,True,-0.000535,True,0.001614,True,0.000888,True
6,HCC7NR,NR,2453,2453,0.010254,0.029231,-0.070682,0.032413,-0.028015,0.010254,...,0.000000,True,0.000000,True,0.000000,True,0.000000,True,0.000000,True


,metric,fraction_direction_preserved
0,C_exh_cyto,1.000000
1,C_exh_tam,0.857143
2,C_malig_caf,1.000000
3,C_b_cyto,1.000000
4,SICI,1.000000


Wrote: latest_figures/Table_S_malignant_restricted.csv


In [11]:
# ----------------------------
# Q6/W10) Benchmark vs Moran's I / Lee's L / Ripley's K
# ----------------------------

import importlib
import subprocess
import sys

for pkg in ["esda", "libpysal"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import numpy as np
import pandas as pd
import squidpy as sq
from scipy.sparse import csr_matrix
from scipy.spatial import cKDTree
from esda import Moran, Moran_BV
from libpysal.weights import WSP

if "st_by_sample" not in globals() or "sample_groups" not in globals():
    raise RuntimeError("Run the main core pipeline cell first to create st_by_sample/sample_groups.")

if "label_permutation_test" not in globals() or "effect_sizes_by_group" not in globals() or "add_bh_fdr" not in globals():
    raise RuntimeError("Run the updated permutation/effect-size cell first.")

program_cols = [
    c for c in [
        "spot_exhaustion",
        "spot_cytotoxic",
        "spot_tam",
        "spot_caf",
        "spot_malignant",
        "spot_bcell",
        "spot_endo",
    ]
    if c in st_adata.obs.columns
]

if not program_cols:
    raise RuntimeError("No expected program columns found for benchmarking.")

sici_pair_specs = [
    ("LEE_EXH_CYTO", "spot_exhaustion", "spot_cytotoxic"),
    ("LEE_MAL_CAF", "spot_malignant", "spot_caf"),
    ("LEE_B_CYTO", "spot_bcell", "spot_cytotoxic"),
]

def _row_standardized_w_from_sparse(sp):
    w = WSP(csr_matrix(sp)).to_W()
    w.transform = "R"
    return w

def _safe_moran(x, sp):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() < 10:
        return np.nan
    sp_sub = csr_matrix(sp[valid][:, valid])
    if sp_sub.nnz == 0:
        return np.nan
    try:
        w = _row_standardized_w_from_sparse(sp_sub)
        return float(Moran(x[valid], w, permutations=0).I)
    except Exception:
        return np.nan

def _safe_moran_bv(x, y, sp):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return np.nan
    sp_sub = csr_matrix(sp[valid][:, valid])
    if sp_sub.nnz == 0:
        return np.nan
    try:
        w = _row_standardized_w_from_sparse(sp_sub)
        return float(Moran_BV(x[valid], y[valid], w, permutations=0).I)
    except Exception:
        return np.nan

def _ripley_k_highspots(ad, value_col, q=0.75):
    if "spatial" not in ad.obsm:
        return np.nan

    vals = ad.obs[value_col].to_numpy(dtype=float)
    coords = np.asarray(ad.obsm["spatial"], dtype=float)
    valid = np.isfinite(vals) & np.isfinite(coords).all(axis=1)
    vals = vals[valid]
    coords = coords[valid]

    if len(vals) < 10:
        return np.nan

    thr = np.nanquantile(vals, q)
    pts = coords[vals > thr]
    n = len(pts)
    if n < 5:
        return np.nan

    tree_all = cKDTree(coords)
    nn_dist, _ = tree_all.query(coords, k=2)
    r = float(np.nanmedian(nn_dist[:, 1]))
    if not np.isfinite(r) or r <= 0:
        return np.nan

    tree_pts = cKDTree(pts)
    n_pairs = len(tree_pts.query_pairs(r))

    x_range = float(np.nanmax(coords[:, 0]) - np.nanmin(coords[:, 0]))
    y_range = float(np.nanmax(coords[:, 1]) - np.nanmin(coords[:, 1]))
    area = x_range * y_range
    if not np.isfinite(area) or area <= 0:
        return np.nan

    return float(area * (2.0 * n_pairs) / (n * (n - 1)))

bench_rows = []
for sample, ad in st_by_sample.items():
    grp = sample_groups.get(sample, None)

    if "spatial_connectivities" not in ad.obsp:
        sq.gr.spatial_neighbors(ad, coord_type="grid")
    sp = ad.obsp["spatial_connectivities"]

    for prog in program_cols:
        m_i = _safe_moran(ad.obs[prog].to_numpy(dtype=float), sp)
        r_k = _ripley_k_highspots(ad, prog, q=0.75)
        bench_rows.append(
            {
                "sample": sample,
                "group": grp,
                "stat_family": "moran_i",
                "metric": f"MORAN_{prog.upper().replace('SPOT_', '')}",
                "value": m_i,
            }
        )
        bench_rows.append(
            {
                "sample": sample,
                "group": grp,
                "stat_family": "ripley_k",
                "metric": f"RIPLEYK_{prog.upper().replace('SPOT_', '')}",
                "value": r_k,
            }
        )

    for name, x_col, y_col in sici_pair_specs:
        lee_l = _safe_moran_bv(
            ad.obs[x_col].to_numpy(dtype=float),
            ad.obs[y_col].to_numpy(dtype=float),
            sp,
        )
        bench_rows.append(
            {
                "sample": sample,
                "group": grp,
                "stat_family": "lee_l_global",
                "metric": name,
                "value": lee_l,
            }
        )

benchmark_stats_df = pd.DataFrame(bench_rows)
benchmark_stats_df = benchmark_stats_df[benchmark_stats_df["group"].isin(["R", "NR"])].copy()
display(benchmark_stats_df.head())

N_PERM_BENCH = int(globals().get("N_PERM_LABEL_DETAILED", 20000))
bench_perm_rows = []
for metric, sub in benchmark_stats_df.groupby("metric"):
    dsub = sub[["sample", "group", "value"]].dropna().copy()
    if dsub.empty or set(dsub["group"].unique()) != {"R", "NR"}:
        continue

    obs, p_perm, _ = label_permutation_test(
        dsub.rename(columns={"value": "metric_value"}),
        value_col="metric_value",
        group_col="group",
        n_perm=N_PERM_BENCH,
        seed=1,
    )
    eff = effect_sizes_by_group(
        dsub.rename(columns={"value": "metric_value"}),
        value_col="metric_value",
        group_col="group",
    )
    bench_perm_rows.append(
        {
            "metric": metric,
            "stat_family": sub["stat_family"].iloc[0],
            "diff_mean_R_minus_NR": obs,
            "p_perm": p_perm,
            "cohens_d": eff["cohens_d"],
            "cliffs_delta": eff["cliffs_delta"],
            "mannwhitney_u": eff["mannwhitney_u"],
            "p_mannwhitney": eff["p_mannwhitney"],
            "C_all": eff["C_all"],
        }
    )

benchmark_perm_df = pd.DataFrame(bench_perm_rows)
if not benchmark_perm_df.empty:
    benchmark_perm_df = add_bh_fdr(
        benchmark_perm_df,
        p_col="p_perm",
        q_col="q_fdr_bh",
        reject_col="reject_fdr_0p05",
        alpha=0.05,
    )
    benchmark_perm_df = benchmark_perm_df.sort_values(["p_perm", "metric"]).reset_index(drop=True)

display(benchmark_perm_df)

comparison_rows = []
for _, r in benchmark_perm_df.iterrows():
    comparison_rows.append(
        {
            "metric": r["metric"],
            "family": r["stat_family"],
            "diff_mean_R_minus_NR": r["diff_mean_R_minus_NR"],
            "cohens_d": r["cohens_d"],
            "cliffs_delta": r["cliffs_delta"],
            "p_perm": r["p_perm"],
            "q_fdr_bh": r["q_fdr_bh"],
        }
    )

# Optional: include field-energy features if present in this notebook context.
if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame):
    energy_cols = [c for c in metrics_df.columns if c.startswith("E_")]
    for e_col in energy_cols:
        dsub = metrics_df[["sample", "group", e_col]].dropna().copy()
        if dsub.empty or set(dsub["group"].unique()) != {"R", "NR"}:
            continue

        obs_e, p_e, _ = label_permutation_test(
            dsub.rename(columns={e_col: "metric_value"}),
            value_col="metric_value",
            group_col="group",
            n_perm=N_PERM_BENCH,
            seed=1,
        )
        eff_e = effect_sizes_by_group(
            dsub.rename(columns={e_col: "metric_value"}),
            value_col="metric_value",
            group_col="group",
        )
        comparison_rows.append(
            {
                "metric": e_col,
                "family": "field_energy",
                "diff_mean_R_minus_NR": obs_e,
                "cohens_d": eff_e["cohens_d"],
                "cliffs_delta": eff_e["cliffs_delta"],
                "p_perm": p_e,
                "q_fdr_bh": np.nan,
            }
        )

if "perm_df" in globals() and isinstance(perm_df, pd.DataFrame):
    coupling_compare_metrics = ["SICI", "C_exh_cyto", "C_b_cyto", "C_malig_caf"]
    for _, r in perm_df[perm_df["metric"].isin(coupling_compare_metrics)].iterrows():
        comparison_rows.append(
            {
                "metric": r["metric"],
                "family": "coupling_or_sici",
                "diff_mean_R_minus_NR": r.get("diff_mean_R_minus_NR", np.nan),
                "cohens_d": r.get("cohens_d", np.nan),
                "cliffs_delta": r.get("cliffs_delta", np.nan),
                "p_perm": r.get("p_perm", np.nan),
                "q_fdr_bh": r.get("q_fdr_bh", np.nan),
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df = comparison_df.sort_values(["family", "q_fdr_bh", "metric"], na_position="last").reset_index(drop=True)

display(comparison_df)

out_dir_local = globals().get("out_dir", "latest_figures")
import os
os.makedirs(out_dir_local, exist_ok=True)
benchmark_stats_df.to_csv(f"{out_dir_local}/Table_S_benchmark_spatial_stats_by_sample.csv", index=False)
benchmark_perm_df.to_csv(f"{out_dir_local}/Table_S_benchmark_moran_lee_ripley_perm.csv", index=False)
comparison_df.to_csv(f"{out_dir_local}/Table_S_benchmark_vs_coupling_effectsizes.csv", index=False)

print("Wrote:")
print(f" - {out_dir_local}/Table_S_benchmark_spatial_stats_by_sample.csv")
print(f" - {out_dir_local}/Table_S_benchmark_moran_lee_ripley_perm.csv")
print(f" - {out_dir_local}/Table_S_benchmark_vs_coupling_effectsizes.csv")


('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/libpysal/weights/weights.py:1683: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 1977.
  w = W(neighbors, weights, ids, silence_warnings=silence_warnings)


('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 1977, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/libpysal/weights/weights.py:1683: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There is 1 island with id: 2262.
  w = W(neighbors, weights, ids, silence_warnings=silence_warnings)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/libpysal/weights/weights.py:1683: UserWarning: The weights matrix is not fully connected: 
 There are 5 disconnected components.
 There are 3 islands with ids: 691, 1633, 1874.
  w = W(neighbors, weights, ids, silence_warnings=silence_warnings)


('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 2262, ' is an island (no neighbors)')
('WARNING: ', 691, ' is an island (no neighbors)')
('WARNING: ', 1633, ' is an island (no neighbors)')
('WARNING: ', 1874, ' is an island (no neighbors)')
('WARNING: ', 691, ' is an island (no neighbors)')
('WARNING: ', 1633, ' is an island (no neighbors)')
('WARNING: ', 1874, ' is an island (no neighbors)')
('WARNING: ', 691, ' is an island (no neighbors)')
('WARNING: ', 1633, ' is an island (no neighbors)')
('WARNING: ', 1874, ' is an island (no neighbors)')
('WARNING: ', 691, ' is an island (no neighbors)')
('WARNING: ', 16

...

('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')
('WARNING: ', 1941, ' is an island (no neighbors)')
('WARNING: ', 2083, ' is an island (no neighbors)')


,sample,group,stat_family,metric,value
0,HCC1R,R,moran_i,MORAN_EXHAUSTION,0.011318
1,HCC1R,R,ripley_k,RIPLEYK_EXHAUSTION,1649.575300
2,HCC1R,R,moran_i,MORAN_CYTOTOXIC,0.041743
3,HCC1R,R,ripley_k,RIPLEYK_CYTOTOXIC,1283.607766
4,HCC1R,R,moran_i,MORAN_TAM,0.409902


,metric,stat_family,diff_mean_R_minus_NR,p_perm,cohens_d,cliffs_delta,mannwhitney_u,p_mannwhitney,C_all,q_fdr_bh,reject_fdr_0p05
0,LEE_MAL_CAF,lee_l_global,0.126914,0.054647,1.927406,1.000000,12.0,0.057143,-0.125305,0.609579,False
1,MORAN_CAF,moran_i,-0.192356,0.110044,-1.539741,-0.666667,2.0,0.228571,0.549131,0.609579,False
2,RIPLEYK_EXHAUSTION,ripley_k,870.876720,0.229689,1.031016,0.666667,10.0,0.199090,941.859558,0.609579,False
3,MORAN_CYTOTOXIC,moran_i,0.056802,0.284336,0.919017,0.500000,9.0,0.400000,0.049578,0.609579,False
4,MORAN_ENDO,moran_i,0.024005,0.312434,0.783826,0.333333,8.0,0.628571,0.064679,0.609579,False
5,RIPLEYK_BCELL,ripley_k,948.299572,0.313984,0.872010,0.500000,9.0,0.359012,1123.526973,0.609579,False
6,RIPLEYK_CYTOTOXIC,ripley_k,700.941135,0.315434,0.769820,0.500000,9.0,0.359012,951.566584,0.609579,False
7,MORAN_BCELL,moran_i,0.153598,0.346383,0.868699,0.500000,9.0,0.400000,0.277612,0.609579,False
8,LEE_EXH_CYTO,lee_l_global,0.029660,0.370231,0.839042,0.500000,9.0,0.400000,0.035881,0.609579,False
9,LEE_B_CYTO,lee_l_global,0.073052,0.458727,0.751302,0.500000,9.0,0.400000,0.093200,0.609579,False


,metric,family,diff_mean_R_minus_NR,cohens_d,cliffs_delta,p_perm,q_fdr_bh
0,C_malig_caf,coupling_or_sici,0.038223,2.401662,1.000000,0.026949,0.477876
1,C_exh_cyto,coupling_or_sici,0.029606,1.231222,0.666667,0.197890,0.655424
2,SICI,coupling_or_sici,0.082107,1.320627,0.833333,0.138593,0.655424
3,C_b_cyto,coupling_or_sici,0.014278,0.326959,0.166667,0.747413,0.877615
4,LEE_B_CYTO,lee_l_global,0.073052,0.751302,0.500000,0.458727,0.609579
5,LEE_EXH_CYTO,lee_l_global,0.029660,0.839042,0.500000,0.370231,0.609579
6,LEE_MAL_CAF,lee_l_global,0.126914,1.927406,1.000000,0.054647,0.609579
7,MORAN_BCELL,moran_i,0.153598,0.868699,0.500000,0.346383,0.609579
8,MORAN_CAF,moran_i,-0.192356,-1.539741,-0.666667,0.110044,0.609579
9,MORAN_CYTOTOXIC,moran_i,0.056802,0.919017,0.500000,0.284336,0.609579


Wrote:
 - latest_figures/Table_S_benchmark_spatial_stats_by_sample.csv
 - latest_figures/Table_S_benchmark_moran_lee_ripley_perm.csv
 - latest_figures/Table_S_benchmark_vs_coupling_effectsizes.csv
